**Trabajo Final:** Afluencia de pasajeros del Metro (EDA)
**Asignatura:** Big Data

**Integrantes:**
- Milena Estrada Muñoz / 1.001.035.917
- Yuliana Estrada Calle / 1.007.253.661
- Isis Valentina Palacio Taborda / 1.000.306.896
- Maria Antonia Pino Giraldo / 1.000.760.444

# Análisis EDA — Afluencia de pasajeros del Metro (arquitectura Medallion)

**Fuente:** `/Volumes/workspace/entrega/data_metro/Data metro.xlsx` (hoja `Hoja1`)

El notebook construye la arquitectura **Medallion** en el schema `workspace.entrega` (mismo schema del volumen):

| Capa | Tabla | Contenido |
|---|---|---|
| **Bronze** | `workspace.entrega.bronze_metro_pasajeros_excel` | Copia cruda del Excel: matriz ancha original, todo como texto |
| **Silver** | `workspace.entrega.silver_metro_pasajeros` | Datos limpios y tidios: `fecha`, `linea`, `hora`, `pasajeros` |
| **Gold** | `workspace.entrega.gold_metro_pasajeros_diarios`, `gold_metro_pasajeros_hora`, `gold_metro_tendencia`, `gold_metro_pronostico_30d` | Datos enriquecidos: agregados diarios, perfil por hora, tendencia y pronóstico |

Entregables del análisis:
1. EDA: calidad de datos, resumen estadístico y perfil de afluencia por hora
2. Pasajeros por día, hora y línea
3. Predicción por línea: si la afluencia **aumenta, disminuye o se mantiene estable** (regresión lineal + estacionalidad semanal, registrada en MLflow)
4. Visualización en **serie de tiempo** con pronóstico a 30 días

In [0]:
%pip install openpyxl plotly
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 148.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Ruta del archivo fuente (volumen de Unity Catalog)
RUTA_EXCEL = "/Volumes/workspace/default/metro/Data metro.xlsx"
HOJA = "Hoja1"

# Tablas de la arquitectura Medallion (en el mismo schema del volumen: workspace.entrega)
TABLA_BRONZE = "workspace.entrega.bronze_metro_pasajeros_excel"
TABLA_SILVER = "workspace.entrega.silver_metro_pasajeros"
TABLA_GOLD_DIARIOS = "workspace.entrega.gold_metro_pasajeros_diarios"
TABLA_GOLD_HORA = "workspace.entrega.gold_metro_pasajeros_hora"
TABLA_GOLD_TENDENCIA = "workspace.entrega.gold_metro_tendencia"
TABLA_GOLD_PRONOSTICO = "workspace.entrega.gold_metro_pronostico_30d"

# Reglas de negocio
DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
UMBRAL_ESTABILIDAD = 0.02   # ±2% en la proyección a 30 días => se considera ESTABLE
HORIZONTE_PRONOSTICO = 30   # días

print("Configuración lista")

Configuración lista


## 🥉 Bronze — Carga de datos crudos
Se lee el Excel del volumen y se guarda **tal cual está** (matriz ancha original, horas en filas, fechas+líneas en columnas), con todo convertido a texto y metadatos de ingesta. Esta capa es la copia fiel del origen.

In [0]:
import pandas as pd

# BRONZE: copia cruda del Excel — se preserva la matriz ancha original, todo como texto
raw = pd.read_excel(RUTA_EXCEL, sheet_name=HOJA, dtype=object)
raw.columns = [str(c) for c in raw.columns]   # nombres de columna normalizados a texto
raw = raw.astype(str)                          # valores crudos como texto (fiel al origen)

bronze_df = (spark.createDataFrame(raw)
    .withColumn("archivo_origen", F.lit("Data metro.xlsx"))
    .withColumn("hoja_origen", F.lit(HOJA))
    .withColumn("fecha_ingesta", F.current_timestamp()))

(bronze_df.write.mode("overwrite").option("overwriteSchema", "true")
    .option("delta.columnMapping.mode", "name")   # permite conservar nombres originales con espacios y ':'
    .saveAsTable(TABLA_BRONZE))

print(f"BRONZE -> {TABLA_BRONZE}: {bronze_df.count()} filas x {len(bronze_df.columns)} columnas")

bronze_tbl = spark.table(TABLA_BRONZE)
display(bronze_tbl.select("Día", "`Unnamed: 1`", *[f"`{c}`" for c in bronze_tbl.columns[2:10]]))

BRONZE -> workspace.entrega.bronze_metro_pasajeros_excel: 22 filas x 429 columnas


Día,Unnamed: 1,2026-01-01 00:00:00,2026-01-01 00:00:00.1,2026-01-02 00:00:00,2026-01-02 00:00:00.1,2026-01-03 00:00:00,2026-01-03 00:00:00.1,2026-01-04 00:00:00,2026-01-04 00:00:00.1
Línea de Servicio,nan,LÍNEA A,LÍNEA B,LÍNEA A,LÍNEA B,LÍNEA A,LÍNEA B,LÍNEA A,LÍNEA B
Hora de operación,04:00:00,1955,351,8054,1479,8104,1493,3570,610
nan,05:00:00,5775,771,16795,2351,15306,2202,9211,1296
nan,06:00:00,5710,850,21941,3241,17718,2657,9572,1284
nan,07:00:00,4390,626,21618,3299,18706,2750,8579,1310
nan,08:00:00,4072,674,16235,2539,15803,2602,8609,1603
nan,09:00:00,4367,857,15711,2610,16161,2833,10385,2055
nan,10:00:00,5684,1109,15966,2883,17164,3160,11617,2301
nan,11:00:00,7057,1358,17751,3088,20104,3586,12615,2566
nan,12:00:00,8046,1598,18933,3106,25914,4045,14251,2925


## 🥈 Silver — Limpieza y transformación
La matriz ancha se convierte a formato **tidy** (`fecha`, `linea`, `hora`, `pasajeros`):
- Cada par de columnas por fecha (`fecha` = LÍNEA A, `fecha.1` = LÍNEA B) se transforma en filas.
- Se conservan solo las filas de hora (04:00–23:00); la fila `Total general` es derivable y se usa solo para validar.
- Se tipifican las columnas: `fecha` (date), `hora` (int), `pasajeros` (int).

In [0]:
# SILVER: transformación a formato tidy (fecha, linea, hora, pasajeros)
bronze_tbl = spark.table(TABLA_BRONZE)
meta = {"Día", "Unnamed: 1", "archivo_origen", "hoja_origen", "fecha_ingesta"}
data_cols = [c for c in bronze_tbl.columns if c not in meta]
cols_a = [c for c in data_cols if not c.endswith(".1")]   # LÍNEA A
cols_b = [c for c in data_cols if c.endswith(".1")]        # LÍNEA B
assert len(cols_a) == len(cols_b) and len(cols_a) > 0, "Columnas A/B no están emparejadas"
assert all(c + ".1" in cols_b for c in cols_a), "Cada columna de LÍNEA A debe tener su par de LÍNEA B"

def unpivot(cols_list, linea):
    """Convierte las columnas de fechas de una línea en filas (fecha, valor)."""
    entries = ", ".join(f"to_date('{c[:10]}'), `{c}`" for c in cols_list)
    return (bronze_tbl
            .select(F.col("`Unnamed: 1`").alias("hora_label"),
                    F.expr(f"stack({len(cols_list)}, {entries}) as (fecha, valor)"))
            .withColumn("linea", F.lit(linea)))

silver_df = (unpivot(cols_a, "A").unionByName(unpivot(cols_b, "B"))
    # solo filas de hora (04:00–23:00); descarta la fila 'Total general', que es derivable
    .where(F.col("hora_label").rlike("^[0-9]{2}:[0-9]{2}:[0-9]{2}$"))
    .withColumn("hora", F.hour(F.to_timestamp("hora_label")))
    .withColumn("pasajeros", F.col("valor").cast("int"))
    .where(F.col("pasajeros").isNotNull())
    .select("fecha", "linea", "hora", "pasajeros"))

(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER))

print(f"SILVER -> {TABLA_SILVER}: {silver_df.count()} filas (esperadas: {len(cols_a) * 20 * 2})")
display(spark.table(TABLA_SILVER).orderBy("fecha", "linea", "hora").limit(10))

SILVER -> workspace.entrega.silver_metro_pasajeros: 8480 filas (esperadas: 8480)


fecha,linea,hora,pasajeros
2026-01-01,A,4,1955
2026-01-01,A,5,5775
2026-01-01,A,6,5710
2026-01-01,A,7,4390
2026-01-01,A,8,4072
2026-01-01,A,9,4367
2026-01-01,A,10,5684
2026-01-01,A,11,7057
2026-01-01,A,12,8046
2026-01-01,A,13,9135


In [0]:
# Validación de calidad de la capa Silver
silver_tbl = spark.table(TABLA_SILVER)

esperadas = len(cols_a) * 20 * 2
tot_label = "Total general (Número de pasajeros)"

checks = {
    "filas_totales": silver_tbl.count(),
    "filas_esperadas": esperadas,
    "nulos": silver_tbl.where(F.col("pasajeros").isNull()).count(),
    "negativos": silver_tbl.where(F.col("pasajeros") < 0).count(),
    "fechas_distintas": silver_tbl.select("fecha").distinct().count(),
    "lineas": sorted(r.linea for r in silver_tbl.select("linea").distinct().collect()),
    "horas": sorted(r.hora for r in silver_tbl.select("hora").distinct().collect()),
}
rango = silver_tbl.agg(F.min("fecha").alias("min").alias("minimo"), F.max("fecha").alias("maximo")).first()
checks["rango_fechas"] = f"{rango['minimo']} a {rango['maximo']}"

# Cruce contra la fila 'Total general' del Excel original (verificación externa)
totales_excel = (unpivot(cols_a, "A").unionByName(unpivot(cols_b, "B"))
    .where(F.col("hora_label") == tot_label)
    .select("fecha", "linea", F.col("valor").cast("int").alias("total_excel")))

dif = (silver_tbl.groupBy("fecha", "linea")
    .agg(F.sum("pasajeros").alias("total_silver"))
    .join(totales_excel, ["fecha", "linea"])
    .where(F.col("total_silver") != F.col("total_excel")))

checks["dias_que_no_coinciden_con_excel"] = dif.count()

assert checks["filas_totales"] == esperadas, "El número de filas no coincide con lo esperado"
assert checks["nulos"] == 0 and checks["negativos"] == 0, "Hay nulos o negativos en la capa silver"
assert checks["dias_que_no_coinciden_con_excel"] == 0, "La suma por hora no reproduce el total del Excel"
for k, v in checks.items():
    print(f"{k}: {v}")
print("✔ Silver validada correctamente")

filas_totales: 8480
filas_esperadas: 8480
nulos: 0
negativos: 0
fechas_distintas: 212
lineas: ['A', 'B']
horas: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
rango_fechas: 2026-01-01 a 2026-07-31
dias_que_no_coinciden_con_excel: 0
✔ Silver validada correctamente


## 📊 Análisis Exploratorio (EDA)
Resumen estadístico por línea, días/horas pico y perfiles de afluencia por hora y día de la semana.

In [0]:
# EDA: resumen estadístico por línea y día/hora más concurridos de cada línea
resumen = (spark.table(TABLA_SILVER).groupBy("linea").agg(
    F.count("pasajeros").alias("observaciones"),
    F.sum("pasajeros").alias("total_pasajeros_periodo"),
    F.round(F.avg("pasajeros"), 1).alias("media_por_hora"),
    F.round(F.stddev("pasajeros"), 1).alias("desviacion_por_hora"),
    F.min("pasajeros").alias("min_por_hora"),
    F.max("pasajeros").alias("max_por_hora")))
display(resumen)

picos = (spark.table(TABLA_SILVER).groupBy("linea", "fecha", "hora")
    .agg(F.sum("pasajeros").alias("pasajeros")))
w = Window.partitionBy("linea").orderBy(F.desc("pasajeros"))
display(picos.withColumn("rank", F.row_number().over(w)).where(F.col("rank") == 1).drop("rank"))

linea,observaciones,total_pasajeros_periodo,media_por_hora,desviacion_por_hora,min_por_hora,max_por_hora
A,4240,119709152,28233.3,18871.5,0,89036
B,4240,17244234,4067.0,2433.9,0,11766


linea,fecha,hora,pasajeros
A,2026-03-03,17,89036
B,2026-03-25,17,11766


In [0]:
import plotly.express as px

# Perfil por hora: promedio de pasajeros por hora y línea
perfil = (spark.table(TABLA_SILVER).groupBy("linea", "hora")
    .agg(F.round(F.avg("pasajeros"), 0).alias("promedio_pasajeros")).toPandas())

fig = px.bar(perfil, x="hora", y="promedio_pasajeros", color="linea", barmode="group",
             labels={"hora": "Hora del día", "promedio_pasajeros": "Promedio de pasajeros", "linea": "Línea"},
             title="Promedio de pasajeros por hora y línea (ene–jul 2026)")
fig.show()

# Mapa de calor: día de la semana × hora
hm = (spark.table(TABLA_SILVER)
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]),
                                           ((F.dayofweek("fecha") + 5) % 7) + 1))
    .groupBy("dia_semana", "hora").agg(F.round(F.avg("pasajeros"), 0).alias("promedio")).toPandas())
pivot_hm = hm.pivot(index="hora", columns="dia_semana", values="promedio").reindex(columns=DIAS_SEMANA)

fig2 = px.imshow(pivot_hm, labels=dict(x="Día de la semana", y="Hora", color="Promedio"),
                 title="Mapa de calor: afluencia promedio por día de la semana y hora (ambas líneas)")
fig2.show()

<!doctype html>

<!doctype html>

## 🥇 Gold — Datos enriquecidos
1. **`metro_pasajeros_diarios`**: total diario por línea + hora pico, día de la semana, fin de semana, mes, promedio móvil de 7 días y total del sistema.
2. **`metro_pasajeros_hora`**: perfil típico por hora y línea (promedio, total y participación de cada hora en el día).

In [0]:
# GOLD 1: agregado diario enriquecido por línea
agg = (spark.table(TABLA_SILVER).groupBy("fecha", "linea")
    .agg(F.sum("pasajeros").alias("total_pasajeros"),
         F.round(F.avg("pasajeros"), 1).alias("promedio_por_hora"),
         F.max(F.struct(F.col("pasajeros").alias("p"), F.col("hora").alias("h"))).alias("pico"))
    .select("fecha", "linea", "total_pasajeros", "promedio_por_hora",
            F.col("pico.h").alias("hora_pico"), F.col("pico.p").alias("pasajeros_hora_pico")))

gold_diarios = (agg
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]),
                                            ((F.dayofweek("fecha") + 5) % 7) + 1))
    .withColumn("fin_de_semana", F.col("dia_semana").isin("Sábado", "Domingo"))
    .withColumn("mes", F.date_format("fecha", "yyyy-MM")))

w_linea = Window.partitionBy("linea").orderBy("fecha")
w_fecha = Window.partitionBy("fecha")
gold_diarios = (gold_diarios
    .withColumn("promedio_movil_7d", F.round(F.avg("total_pasajeros").over(w_linea.rowsBetween(-6, 0)), 0))
    .withColumn("total_sistema", F.sum("total_pasajeros").over(w_fecha)))

(gold_diarios.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_DIARIOS))

print(f"GOLD -> {TABLA_GOLD_DIARIOS}: {gold_diarios.count()} filas")
display(spark.table(TABLA_GOLD_DIARIOS).orderBy(F.desc("fecha"), "linea").limit(10))

GOLD -> workspace.entrega.gold_metro_pasajeros_diarios: 424 filas


fecha,linea,total_pasajeros,promedio_por_hora,hora_pico,pasajeros_hora_pico,dia_semana,fin_de_semana,mes,promedio_movil_7d,total_sistema
2026-07-31,A,688259,34413.0,16,67716,Viernes,false,2026-07,579960.0,785344
2026-07-31,B,97085,4854.3,16,8438,Viernes,false,2026-07,83638.0,785344
2026-07-30,A,671672,33583.6,17,74144,Jueves,false,2026-07,577822.0,766318
2026-07-30,B,94646,4732.3,17,9383,Jueves,false,2026-07,83247.0,766318
2026-07-29,A,674878,33743.9,17,75429,Miércoles,false,2026-07,577409.0,770243
2026-07-29,B,95365,4768.3,17,9555,Miércoles,false,2026-07,83053.0,770243
2026-07-28,A,670027,33501.4,17,75932,Martes,false,2026-07,576351.0,764215
2026-07-28,B,94188,4709.4,17,9561,Martes,false,2026-07,83370.0,764215
2026-07-27,A,651454,32572.7,17,75918,Lunes,false,2026-07,576789.0,742445
2026-07-27,B,90991,4549.6,17,9580,Lunes,false,2026-07,83800.0,742445


In [0]:
# GOLD 2: perfil por hora y línea — cuántos pasajeros transitan por hora en cada línea
w_linea = Window.partitionBy("linea")
gold_hora = (spark.table(TABLA_SILVER).groupBy("linea", "hora")
    .agg(F.round(F.avg("pasajeros"), 0).alias("promedio_pasajeros"),
         F.sum("pasajeros").alias("total_periodo"),
         F.round(F.avg(F.when(F.col("pasajeros") > 0, F.col("pasajeros"))), 0).alias("promedio_horas_con_servicio"))
    .withColumn("participacion_del_dia", F.round(
        F.col("promedio_pasajeros") / F.sum("promedio_pasajeros").over(w_linea), 4))
    .orderBy("linea", "hora"))

(gold_hora.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_HORA))

print(f"GOLD -> {TABLA_GOLD_HORA}: {gold_hora.count()} filas")
display(gold_hora)

GOLD -> workspace.entrega.gold_metro_pasajeros_hora: 40 filas


linea,hora,promedio_pasajeros,total_periodo,promedio_horas_con_servicio,participacion_del_dia
A,4,12685.0,2689155,12685.0,0.0225
A,5,35552.0,7536965,35552.0,0.063
A,6,45787.0,9706807,45787.0,0.0811
A,7,40647.0,8617225,40647.0,0.072
A,8,26905.0,5703918,26905.0,0.0476
A,9,23278.0,4934931,23278.0,0.0412
A,10,21719.0,4604451,21719.0,0.0385
A,11,23091.0,4895388,23091.0,0.0409
A,12,27832.0,5900469,27832.0,0.0493
A,13,28296.0,5998704,28296.0,0.0501


## 🔮 Predicción de tendencia y serie de tiempo
Por cada línea se ajusta una **regresión lineal** sobre los totales diarios:
- `proyeccion_30d_pct` = variación esperada en 30 días respecto al promedio diario.
- Clasificación: **AUMENTA** (> +2%), **DISMINUYE** (< −2%) o **ESTABLE** (entre ±2%).
- El experimento se registra en **MLflow** (métricas y modelos por línea).

El pronóstico a 30 días combina la **tendencia** con el **factor estacional de cada día de la semana**.

In [0]:
import mlflow
import pandas as pd
from sklearn.linear_model import LinearRegression
from mlflow.models import infer_signature

# Datos diarios por línea (gold)
diarios = spark.table(TABLA_GOLD_DIARIOS).select("fecha", "linea", "total_pasajeros").toPandas()
diarios["fecha"] = pd.to_datetime(diarios["fecha"])
fecha_max = diarios["fecha"].max()

resultados, modelos = [], {}

mlflow.set_experiment("/Users/yuliestrada12@gmail.com/metro-tendencia-pasajeros")
with mlflow.start_run(run_name="tendencia_lineas_metro"):
    mlflow.log_param("umbral_estabilidad_pct", UMBRAL_ESTABILIDAD * 100)
    mlflow.log_param("horizonte_dias", HORIZONTE_PRONOSTICO)
    for linea in sorted(diarios["linea"].unique()):
        g = diarios[diarios["linea"] == linea].sort_values("fecha")
        x = ((g["fecha"] - g["fecha"].min()).dt.days).values.reshape(-1, 1)
        y = g["total_pasajeros"].values.astype(float)
        reg = LinearRegression().fit(x, y)
        pendiente = float(reg.coef_[0])
        proyeccion_30d_pct = pendiente * 30 / y.mean()
        clasificacion = ("AUMENTA" if proyeccion_30d_pct > UMBRAL_ESTABILIDAD
                         else "DISMINUYE" if proyeccion_30d_pct < -UMBRAL_ESTABILIDAD
                         else "ESTABLE")
        # Cambio reciente: últimos 30 días vs 30 días anteriores
        ult, prev = y[-30:].mean(), y[-60:-30].mean()
        cambio_reciente_pct = (ult - prev) / prev
        mlflow.log_metrics({
            f"{linea}_pendiente_diaria": pendiente,
            f"{linea}_proyeccion_30d_pct": proyeccion_30d_pct * 100,
            f"{linea}_r2": reg.score(x, y),
            f"{linea}_cambio_reciente_30d_pct": cambio_reciente_pct * 100,
        })
        firma = infer_signature(x, reg.predict(x))
        mlflow.sklearn.log_model(reg, name=f"modelo_tendencia_{linea}",
                                 signature=firma, input_example=x[:3])
        resultados.append({"linea": linea, "pendiente_diaria": round(pendiente, 2),
                           "proyeccion_30d_pct": round(proyeccion_30d_pct * 100, 2),
                           "r2": round(reg.score(x, y), 4),
                           "cambio_reciente_30d_pct": round(cambio_reciente_pct * 100, 2),
                           "clasificacion": clasificacion})
        modelos[linea] = (reg, g["fecha"].min())

tendencia_df = spark.createDataFrame(pd.DataFrame(resultados))
(tendencia_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_TENDENCIA))

print(f"GOLD -> {TABLA_GOLD_TENDENCIA}")
display(tendencia_df)

🔗 View Logged Model at: https://dbc-903c9af9-893c.cloud.databricks.com/ml/experiments/4424435423998567/models/m-bc327006553143dc8c15e52f14fc84de?o=7474650957557286
🔗 View Logged Model at: https://dbc-903c9af9-893c.cloud.databricks.com/ml/experiments/4424435423998567/models/m-aa17ea92398d4cb1b6833fec33b760bf?o=7474650957557286


GOLD -> workspace.entrega.gold_metro_tendencia


linea,pendiente_diaria,proyeccion_30d_pct,r2,cambio_reciente_30d_pct,clasificacion
A,-34.09,-0.18,1.0E-4,0.64,ESTABLE
B,-18.15,-0.67,0.002,2.61,ESTABLE


In [0]:
import pandas as pd

# PRONÓSTICO a 30 días: tendencia (regresión) × factor estacional del día de la semana
fechas_futuras = pd.date_range(fecha_max + pd.Timedelta(days=1), periods=HORIZONTE_PRONOSTICO, freq="D")
filas = []
for linea, (reg, fecha_min_linea) in modelos.items():
    g = diarios[diarios["linea"] == linea]
    factores = g.groupby(g["fecha"].dt.dayofweek)["total_pasajeros"].mean() / g["total_pasajeros"].mean()
    for f in fechas_futuras:
        t = (f - fecha_min_linea).days
        base = reg.intercept_ + reg.coef_[0] * t
        factor = factores.get(f.dayofweek, 1.0)
        filas.append({"linea": linea, "fecha": f.date(),
                      "pasajeros_predichos": int(round(max(0.0, base * factor)))})

pronostico_df = spark.createDataFrame(pd.DataFrame(filas))
(pronostico_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_PRONOSTICO))

print(f"GOLD -> {TABLA_GOLD_PRONOSTICO}: {pronostico_df.count()} filas")
display(pronostico_df.orderBy("fecha", "linea").limit(10))

GOLD -> workspace.entrega.gold_metro_pronostico_30d: 60 filas


linea,fecha,pasajeros_predichos
A,2026-08-01,488825
B,2026-08-01,71552
A,2026-08-02,220242
B,2026-08-02,35999
A,2026-08-03,548550
B,2026-08-03,76749
A,2026-08-04,683094
B,2026-08-04,94249
A,2026-08-05,683452
B,2026-08-05,94726


In [0]:
import plotly.graph_objects as go

# SERIE DE TIEMPO: histórico diario, promedio móvil 7 días y pronóstico a 30 días
hist = spark.table(TABLA_GOLD_DIARIOS).select("fecha", "linea", "total_pasajeros", "promedio_movil_7d").toPandas()
hist["fecha"] = pd.to_datetime(hist["fecha"])
pron = spark.table(TABLA_GOLD_PRONOSTICO).toPandas()
pron["fecha"] = pd.to_datetime(pron["fecha"])

fig = go.Figure()
for linea in sorted(hist["linea"].unique()):
    h = hist[hist["linea"] == linea].sort_values("fecha")
    p = pron[pron["linea"] == linea].sort_values("fecha")
    fig.add_trace(go.Scatter(x=h["fecha"], y=h["total_pasajeros"], mode="lines",
                             opacity=0.35, name=f"Línea {linea} (diario)"))
    fig.add_trace(go.Scatter(x=h["fecha"], y=h["promedio_movil_7d"], mode="lines",
                             name=f"Línea {linea} (prom. 7 días)"))
    ultimo = h.iloc[-1]
    fig.add_trace(go.Scatter(
        x=[ultimo["fecha"]] + list(p["fecha"]),
        y=[ultimo["total_pasajeros"]] + list(p["pasajeros_predichos"]),
        mode="lines", line=dict(dash="dash"), name=f"Línea {linea} (pronóstico 30 d)"))

fig.update_layout(title="Serie de tiempo: pasajeros por día y línea — histórico, promedio móvil 7 días y pronóstico",
                  xaxis_title="Fecha", yaxis_title="Pasajeros por día")
fig.show()

<!doctype html>

## ❓ Predicción: preguntas y respuestas

Las preguntas del pronóstico se responden directamente desde las tablas Gold (`gold_metro_tendencia`, `gold_metro_pronostico_30d`, `gold_metro_pasajeros_hora`), de modo que las **respuestas se recalculan automáticamente** en cada ejecución del notebook.

1. ¿La afluencia de pasajeros **aumenta, disminuye o se mantiene estable** en los próximos 30 días?
2. ¿**Cuántos pasajeros** se pronostican en total para los próximos 30 días?
3. ¿Cuáles serán los **días más y menos concurridos** de la semana?
4. ¿Cuál es la **hora pico** de cada línea?

In [0]:
import pandas as pd

# RESPUESTAS calculadas desde las tablas Gold (se recalculan en cada ejecución)
tend = spark.table(TABLA_GOLD_TENDENCIA).toPandas()
pron = spark.table(TABLA_GOLD_PRONOSTICO).toPandas()
pron["fecha"] = pd.to_datetime(pron["fecha"])
hora_tbl = spark.table(TABLA_GOLD_HORA).toPandas()

print("PREGUNTA 1: ¿La afluencia de pasajeros AUMENTA, DISMINUYE o se mantiene ESTABLE en los próximos 30 días?")
for _, r in tend.iterrows():
    print(f"   RESPUESTA — Línea {r['linea']}: {r['clasificacion']} "
          f"(proyección a 30 días: {r['proyeccion_30d_pct']:+.2f}%, "
          f"cambio reciente: {r['cambio_reciente_30d_pct']:+.2f}%, R²: {r['r2']})")

print("\nPREGUNTA 2: ¿Cuántos pasajeros se pronostican en total para los próximos 30 días?")
tot = pron.groupby("linea")["pasajeros_predichos"].sum()
for linea, t in tot.items():
    print(f"   RESPUESTA — Línea {linea}: {t:,} pasajeros ({t / 30:,.0f} por día en promedio)")
print(f"   RESPUESTA — Sistema completo: {tot.sum():,} pasajeros "
      f"({pron['fecha'].min().date()} a {pron['fecha'].max().date()})")

print("\nPREGUNTA 3: ¿Cuáles serán los días más y menos concurridos de la semana?")
for linea in sorted(pron["linea"].unique()):
    g = pron[pron["linea"] == linea]
    por_dia = g.assign(dia=g["fecha"].dt.dayofweek).groupby("dia")["pasajeros_predichos"].mean()
    print(f"   RESPUESTA — Línea {linea}: más concurrido {DIAS_SEMANA[int(por_dia.idxmax())]} "
          f"({por_dia.max():,.0f} promedio), menos concurrido {DIAS_SEMANA[int(por_dia.idxmin())]} "
          f"({por_dia.min():,.0f} promedio)")

print("\nPREGUNTA 4: ¿Cuál es la hora pico de cada línea?")
for linea in sorted(hora_tbl["linea"].unique()):
    h = hora_tbl[hora_tbl["linea"] == linea].nlargest(1, "promedio_pasajeros").iloc[0]
    print(f"   RESPUESTA — Línea {linea}: {int(h['hora'])}:00 hrs "
          f"({h['promedio_pasajeros']:,.0f} pasajeros promedio por hora)")

PREGUNTA 1: ¿La afluencia de pasajeros AUMENTA, DISMINUYE o se mantiene ESTABLE en los próximos 30 días?
   RESPUESTA — Línea A: ESTABLE (proyección a 30 días: -0.18%, cambio reciente: +0.64%, R²: 0.0001)
   RESPUESTA — Línea B: ESTABLE (proyección a 30 días: -0.67%, cambio reciente: +2.61%, R²: 0.002)

PREGUNTA 2: ¿Cuántos pasajeros se pronostican en total para los próximos 30 días?
   RESPUESTA — Línea A: 16,380,325 pasajeros (546,011 por día en promedio)
   RESPUESTA — Línea B: 2,320,223 pasajeros (77,341 por día en promedio)
   RESPUESTA — Sistema completo: 18,700,548 pasajeros (2026-08-01 a 2026-08-30)

PREGUNTA 3: ¿Cuáles serán los días más y menos concurridos de la semana?
   RESPUESTA — Línea A: más concurrido Miércoles (683,016 promedio), menos concurrido Domingo (220,055 promedio)
   RESPUESTA — Línea B: más concurrido Miércoles (94,498 promedio), menos concurrido Domingo (35,884 promedio)

PREGUNTA 4: ¿Cuál es la hora pico de cada línea?
   RESPUESTA — Línea A: 17:00 hrs (58

## 📈 Dashboard global (en el notebook)

Todos los indicadores y gráficos clave del pipeline en **una sola vista**, construidos directamente desde las tablas Gold de `workspace.entrega` (se recalculan automáticamente al re-ejecutar):

- **KPIs**: total histórico, promedio diario del sistema, pronóstico a 30 días y participación de la Línea A.
- **Picos de afluencia** por hora y línea, y afluencia promedio por día de la semana.
- **Serie de tiempo** diaria con el pronóstico a 30 días y **participación por línea**.
- **Mapa de calor** día de la semana × hora y **tabla de tendencia** (¿Aumenta, disminuye o se mantiene?).

In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# DASHBOARD GLOBAL dentro del notebook — se construye desde las tablas Gold en cada ejecución
ORDEN_DIA = {d: i + 1 for i, d in enumerate(DIAS_SEMANA)}
COLORES = {"A": "#1f77b4", "B": "#ff7f0e"}

kpi = spark.sql("""
  SELECT
    (SELECT SUM(total_pasajeros) FROM workspace.entrega.gold_metro_pasajeros_diarios) AS total_historico,
    (SELECT AVG(total_sistema) FROM (SELECT DISTINCT fecha, total_sistema
       FROM workspace.entrega.gold_metro_pasajeros_diarios)) AS promedio_diario,
    (SELECT SUM(pasajeros_predichos) FROM workspace.entrega.gold_metro_pronostico_30d) AS total_pronostico,
    (SELECT SUM(CASE WHEN linea = 'A' THEN total_pasajeros ELSE 0 END) / SUM(total_pasajeros)
       FROM workspace.entrega.gold_metro_pasajeros_diarios) AS share_a
""").toPandas().iloc[0]

perfil_hora = spark.table(TABLA_GOLD_HORA).toPandas()
perfil_dia = (spark.table(TABLA_GOLD_DIARIOS).groupBy("linea", "dia_semana")
    .agg(F.round(F.avg("total_pasajeros"), 0).alias("promedio")).toPandas())
perfil_dia["orden"] = perfil_dia["dia_semana"].map(ORDEN_DIA)
perfil_dia["dia"] = perfil_dia["orden"].astype(str) + ". " + perfil_dia["dia_semana"]

hist = spark.table(TABLA_GOLD_DIARIOS).select("fecha", "linea", "total_pasajeros").toPandas()
hist["fecha"] = pd.to_datetime(hist["fecha"])
pron = spark.table(TABLA_GOLD_PRONOSTICO).toPandas()
pron["fecha"] = pd.to_datetime(pron["fecha"])

heat = (spark.table(TABLA_SILVER)
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]),
                                            ((F.dayofweek("fecha") + 5) % 7) + 1))
    .groupBy("dia_semana", "hora").agg(F.round(F.avg("pasajeros"), 0).alias("promedio")).toPandas())
z = heat.pivot(index="dia_semana", columns="hora", values="promedio").reindex(DIAS_SEMANA)

tend = spark.table(TABLA_GOLD_TENDENCIA).toPandas()

fig = make_subplots(
    rows=4, cols=4, row_heights=[0.14, 0.28, 0.30, 0.28],
    vertical_spacing=0.07, horizontal_spacing=0.05,
    subplot_titles=["", "", "", "",
                    "Picos de afluencia por hora y línea", "Afluencia promedio por día de la semana",
                    "Serie de tiempo: histórico + pronóstico 30 días", "Participación por línea (histórico)",
                    "Mapa de calor: día de la semana × hora", "Tendencia: ¿Aumenta, disminuye o se mantiene?"],
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"colspan": 2}, None, {"colspan": 2}, None],
        [{"colspan": 3}, None, None, {"type": "domain"}],
        [{"colspan": 2}, None, {"colspan": 2, "type": "table"}, None],
    ])

# --- Fila 1: KPIs ---
kpis = [
    ("Total pasajeros (ene–jul 2026)", kpi["total_historico"], ",d"),
    ("Promedio diario del sistema", kpi["promedio_diario"], ",.0f"),
    ("Pronóstico 30 días (ago 2026)", kpi["total_pronostico"], ",d"),
    ("Participación Línea A", kpi["share_a"], ".1%"),
]
for i, (titulo, valor, fmt) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(mode="number", value=float(valor),
                               number={"valueformat": fmt},
                               title={"text": f"<b>{titulo}</b>"}), row=1, col=i)

# --- Fila 2 (izq): picos por hora y línea ---
for linea in ["A", "B"]:
    g = perfil_hora[perfil_hora["linea"] == linea].sort_values("hora")
    fig.add_trace(go.Bar(x=g["hora"].astype(str), y=g["promedio_pasajeros"], name=f"Línea {linea}",
                         marker_color=COLORES[linea]), row=2, col=1)
fig.update_xaxes(title_text="Hora del día", row=2, col=1)
fig.update_yaxes(title_text="Pasajeros promedio", row=2, col=1)

# --- Fila 2 (der): afluencia por día de la semana ---
for linea in ["A", "B"]:
    g = perfil_dia[perfil_dia["linea"] == linea].sort_values("orden")
    fig.add_trace(go.Bar(x=g["dia"], y=g["promedio"], name=f"Línea {linea}",
                         marker_color=COLORES[linea], showlegend=False), row=2, col=3)
fig.update_xaxes(title_text="Día de la semana", row=2, col=3)
fig.update_yaxes(title_text="Pasajeros promedio/día", row=2, col=3)

# --- Fila 3 (izq): serie de tiempo + pronóstico ---
for linea in ["A", "B"]:
    h = hist[hist["linea"] == linea].sort_values("fecha")
    p = pron[pron["linea"] == linea].sort_values("fecha")
    fig.add_trace(go.Scatter(x=h["fecha"], y=h["total_pasajeros"], mode="lines",
                             line=dict(color=COLORES[linea], width=1.5), opacity=0.4,
                             name=f"Línea {linea} (diario)"), row=3, col=1)
    fig.add_trace(go.Scatter(x=[h["fecha"].iloc[-1]] + list(p["fecha"]),
                             y=[h["total_pasajeros"].iloc[-1]] + list(p["pasajeros_predichos"]),
                             mode="lines", line=dict(color=COLORES[linea], dash="dash"),
                             name=f"Línea {linea} (pronóstico)"), row=3, col=1)
fig.update_xaxes(title_text="Fecha", row=3, col=1)
fig.update_yaxes(title_text="Pasajeros por día", row=3, col=1)

# --- Fila 3 (der): participación por línea ---
tot_linea = hist.groupby("linea")["total_pasajeros"].sum()
fig.add_trace(go.Pie(labels=[f"Línea {l}" for l in tot_linea.index], values=tot_linea.values, hole=0.45,
                     marker={"colors": [COLORES[l] for l in tot_linea.index]},
                     showlegend=False), row=3, col=4)

# --- Fila 4 (izq): heatmap día × hora ---
fig.add_trace(go.Heatmap(z=z.values, x=list(z.columns), y=list(z.index),
                         colorscale="Blues", colorbar={"title": "Promedio"}), row=4, col=1)
fig.update_xaxes(title_text="Hora del día", row=4, col=1)
fig.update_yaxes(title_text="Día de la semana", autorange="reversed", row=4, col=1)

# --- Fila 4 (der): tabla de tendencia ---
fig.add_trace(go.Table(
    header={"values": ["Línea", "Clasificación", "Pendiente diaria", "Proyección 30 d (%)",
                       "Cambio reciente (%)", "R²"],
            "fill_color": "#1f77b4", "font": {"color": "white", "size": 12}, "align": "center"},
    cells={"values": [tend[c] for c in ["linea", "clasificacion", "pendiente_diaria",
                                        "proyeccion_30d_pct", "cambio_reciente_30d_pct", "r2"]],
           "align": "center"}), row=4, col=3)

fig.update_layout(height=1150, width=1350, barmode="group",
                  title_text="<b>Dashboard Global — Metro CDMX: histórico ene–jul 2026 y pronóstico agosto</b>",
                  title_x=0.5, legend={"orientation": "h", "yanchor": "top", "y": -0.02})
fig.show()

<!doctype html>

## 🎯 Conclusión general del análisis

### ¿Qué es este proyecto?
Pipeline de datos con arquitectura **Medallion** (Bronze → Silver → Gold) que ingiere el Excel de afluencia de pasajeros del Metro CDMX (líneas A y B, horas 04:00–23:00, 1 ene – 31 jul 2026, 212 días), lo limpia, enriquece y **pronostica la demanda a 30 días** con registro de modelos en MLflow. Todo es reproducible: al re-ejecutar con nuevos datos, tablas, gráficos, respuestas y este resumen se recalculan.

### Hallazgos principales
1. **Calidad de datos**: 8,480 registros validados sin nulos ni negativos; la capa Silver coincide 100% con el origen.
2. **Perfil temporal**: la demanda sigue un patrón semanal muy marcado — picos dobles mañana (06:00–08:00) y tarde, con **hora pico a las 17:00**; martes y miércoles son los días más cargados y el domingo el de menor afluencia.
3. **Estructura de la demanda**: la **Línea A concentra ~87%** del total del sistema (~7 veces la Línea B).
4. **Tendencia y pronóstico**: ambas líneas son **ESTABLE** a 30 días (variación < ±2%); agosto pronostica ~18.7M pasajeros, consistente con el nivel histórico.

### Implicaciones para la toma de decisiones
- **Operación**: concentrar refuerzos de servicio entre semana en la tarde (16:00–19:00, pico 17:00); usar domingo y las horas nocturnas (>21:00) como ventana de bajo impacto para mantenimiento.
- **Planificación**: la estabilidad de la demanda no justifica ampliación de capacidad a corto plazo; priorizar eficiencia y calidad de servicio.
- **Línea B**: su baja utilización sugiere revisar frecuencias y promocionar su uso antes de invertir en más capacidad.
- **Monitoreo**: re-ejecutar el notebook con datos nuevos y vigilar la clasificación — un cambio a AUMENTA o DISMINUYE (umbral ±2%) debe gatillar replanificación.

### Limitaciones
El R² de la regresión es ≈ 0: la variación diaria la domina el **patrón semanal**, no una tendencia lineal, por lo que el pronóstico es confiable a **30 días** pero no a largo plazo, y el análisis cubre solo las líneas A y B de un semestre.

In [0]:
import pandas as pd

# RESUMEN EJECUTIVO dinámico — se recalcula desde las tablas Gold en cada ejecución
tend = spark.table(TABLA_GOLD_TENDENCIA).toPandas()
pron = spark.table(TABLA_GOLD_PRONOSTICO).toPandas()
pron["fecha"] = pd.to_datetime(pron["fecha"])
hora_tbl = spark.table(TABLA_GOLD_HORA).toPandas()
diarios = spark.table(TABLA_GOLD_DIARIOS).toPandas()

total_hist = int(diarios["total_pasajeros"].sum())
n_dias = diarios["fecha"].nunique()
prom_dia = diarios.drop_duplicates("fecha")["total_sistema"].mean()
share_a = diarios[diarios["linea"] == "A"]["total_pasajeros"].sum() / total_hist

por_hora = hora_tbl.groupby("hora")["promedio_pasajeros"].sum()
pico_hora, pico_hora_val = int(por_hora.idxmax()), por_hora.max()

dia_prom = (diarios.drop_duplicates("fecha")
            .groupby("dia_semana")["total_sistema"].mean().reindex(DIAS_SEMANA))

pron_total = int(pron["pasajeros_predichos"].sum())
pron_prom_dia = pron_total / pron["fecha"].nunique()
delta = (pron_prom_dia / prom_dia - 1) * 100

print("=" * 78)
print("RESUMEN EJECUTIVO — Metro CDMX (recalculado desde las tablas Gold)")
print("=" * 78)
print(f"Alcance: {n_dias} días ({diarios['fecha'].min()} a {diarios['fecha'].max()}), líneas A y B, horas 04–23")
print(f"1. Demanda histórica: {total_hist:,} pasajeros en total ({prom_dia:,.0f} por día en promedio).")
print(f"2. Estructura: la Línea A concentra {share_a:.1%} de la afluencia ({share_a / (1 - share_a):.1f} veces la Línea B).")
print(f"3. Hora pico del sistema: {pico_hora}:00 hrs ({pico_hora_val:,.0f} pasajeros promedio/hora).")
print(f"4. Días: más concurrido {dia_prom.idxmax()} ({dia_prom.max():,.0f}/día); menos concurrido {dia_prom.idxmin()} ({dia_prom.min():,.0f}/día).")
print(f"5. Pronóstico 30 días ({pron['fecha'].min().date()} a {pron['fecha'].max().date()}): {pron_total:,} pasajeros "
      f"({pron_prom_dia:,.0f}/día, {delta:+.1f}% vs promedio histórico).")
for _, r in tend.iterrows():
    print(f"6. Línea {r['linea']}: {r['clasificacion']} "
          f"(proyección 30 d: {r['proyeccion_30d_pct']:+.2f}%, cambio reciente: {r['cambio_reciente_30d_pct']:+.2f}%, R²={r['r2']:.4f})")

print("-" * 78)
print("IMPLICACIONES PARA LA TOMA DE DECISIONES")
if (tend["clasificacion"] == "ESTABLE").all():
    print("• Demanda estable a 30 días: no se justifica ampliar capacidad; enfocarse en eficiencia operativa.")
else:
    print("• La demanda está cambiando: revisar la línea clasificada como AUMENTA/DISMINUYE y replanificar capacidad.")
print(f"• Reforzar servicio entre semana de 16:00 a 19:00 (pico a las {pico_hora}:00); ventana óptima para mantenimiento: {dia_prom.idxmin()} y después de las 21:00.")
print(f"• Revisar la Línea B (solo {1 - share_a:.1%} de la demanda): ajustar frecuencias o promocionar su uso.")
print("• Monitorear en cada ejecución: un cambio de clasificación (±2%) debe gatillar replanificación.")
print("• Limitación: R²≈0 — la variación diaria la domina el patrón semanal; pronóstico confiable a 30 días, no a largo plazo.")

RESUMEN EJECUTIVO — Metro CDMX (recalculado desde las tablas Gold)
Alcance: 212 días (2026-01-01 a 2026-07-31), líneas A y B, horas 04–23
1. Demanda histórica: 136,953,386 pasajeros en total (646,007 por día en promedio).
2. Estructura: la Línea A concentra 87.4% de la afluencia (6.9 veces la Línea B).
3. Hora pico del sistema: 17:00 hrs (65,547 pasajeros promedio/hora).
4. Días: más concurrido Miércoles (785,162/día); menos concurrido Domingo (258,564/día).
5. Pronóstico 30 días (2026-08-01 a 2026-08-30): 18,700,548 pasajeros (623,352/día, -3.5% vs promedio histórico).
6. Línea A: ESTABLE (proyección 30 d: -0.18%, cambio reciente: +0.64%, R²=0.0001)
6. Línea B: ESTABLE (proyección 30 d: -0.67%, cambio reciente: +2.61%, R²=0.0020)
------------------------------------------------------------------------------
IMPLICACIONES PARA LA TOMA DE DECISIONES
• Demanda estable a 30 días: no se justifica ampliar capacidad; enfocarse en eficiencia operativa.
• Reforzar servicio entre semana de 16:0